# CrazeAI Mini — 1M-parameter Colab training and GGUF export

This notebook trains a **~951K-parameter** decoder-only Transformer from scratch on up to **10,000 examples**, exports it to Hugging Face format, converts it to GGUF, quantizes to Q4_K_M, smoke-tests it, and downloads the result.

**Runtime target:** around 10 minutes of training on a working Colab GPU, not a guarantee for the entire notebook. Dataset download, JAX compilation, and GGUF conversion are measured separately. This is a pipeline experiment, not a capable conversational model.

Select **Runtime → Change runtime type → T4 GPU** if available.

In [ ]:
# Check runtime hardware without importing JAX before dependencies are installed.
import os, sys, subprocess
print("Python:", sys.version)
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_VISIBLE = gpu_check.returncode == 0
print("GPU visible:", GPU_VISIBLE)
if GPU_VISIBLE:
    print(gpu_check.stdout[:2500])
else:
    print("No GPU detected. The notebook can run on CPU, but the 10-minute target is unlikely to hold.")

In [ ]:
# Clone the test branch and install dependencies.
%cd /content
!rm -rf CrazeAi llama.cpp
!git clone --depth 1 --branch test https://github.com/CrazeStudio/CrazeAi.git
%cd /content/CrazeAi
%pip install -q optax safetensors tokenizers datasets huggingface_hub
import jax, optax, numpy, tokenizers, datasets
BACKEND = jax.default_backend()
print("JAX:", jax.__version__)
print("Backend:", BACKEND, "| Devices:", jax.devices())
if GPU_VISIBLE and BACKEND != "gpu":
    print("WARNING: NVIDIA GPU is visible but JAX selected CPU. This run may be much slower than the target.")

In [ ]:
# 10,000-example training run. Batch size adapts to the actual JAX backend.
MAX_EXAMPLES = 10000
EPOCHS = 1
SEQ_LEN = 128
BATCH_SIZE = 16 if BACKEND == "gpu" else 2
CHUNK_SIZE = 8 if BACKEND == "gpu" else 1
MAX_MODEL_BYTES = 100_000_000

import os, time, json
os.environ["PYTHONUNBUFFERED"] = "1"
!mkdir -p artifacts
prep_start = time.time()
!python training_v2/prepare_sft.py --max-examples {MAX_EXAMPLES} --output artifacts/sft.jsonl
prep_seconds = time.time() - prep_start
!test -s artifacts/sft.jsonl && wc -l artifacts/sft.jsonl
print(f"Dataset preparation wall time: {prep_seconds:.1f}s")

In [ ]:
# Train the isolated ~951K-parameter model and measure actual training wall time.
train_start = time.time()
!python training_jax_1m/train.py --data artifacts/sft.jsonl --output artifacts/crazeai-1m-jax --epochs {EPOCHS} --seq-len {SEQ_LEN} --batch-size {BATCH_SIZE} --max-examples {MAX_EXAMPLES} --chunk-size {CHUNK_SIZE}
train_seconds = time.time() - train_start
!test -s artifacts/crazeai-1m-jax/params.npz
print(f"Training process wall time, including first-step compilation: {train_seconds:.1f}s")

In [ ]:
# Export and validate the small Hugging Face checkpoint.
export_start = time.time()
!python training_jax_1m/export_hf.py --checkpoint artifacts/crazeai-1m-jax --output artifacts/crazeai-1m
!test -s artifacts/crazeai-1m/model.safetensors
!test -s artifacts/crazeai-1m/config.json
!ls -lh artifacts/crazeai-1m/
print(f"Export wall time: {time.time() - export_start:.1f}s")

In [ ]:
# Build llama.cpp converter/quantizer.
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=OFF
!cmake --build llama.cpp/build --config Release -j2 --target llama-quantize llama-cli
%pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# Convert to F16 GGUF, then quantize to Q4_K_M and verify the actual size.
%cd /content/CrazeAi
convert_start = time.time()
!python /content/llama.cpp/convert_hf_to_gguf.py artifacts/crazeai-1m --outfile artifacts/crazeai-1m-f16.gguf --outtype f16
!test -s artifacts/crazeai-1m-f16.gguf
!/content/llama.cpp/build/bin/llama-quantize artifacts/crazeai-1m-f16.gguf artifacts/crazeai-1m-q4_k_m.gguf Q4_K_M
!test -s artifacts/crazeai-1m-q4_k_m.gguf
import os
model_path = "/content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf"
size_bytes = os.path.getsize(model_path)
print(f"Conversion + quantization wall time: {time.time() - convert_start:.1f}s")
print(f"GGUF size: {size_bytes:,} bytes ({size_bytes / 1_000_000:.2f} MB)")
if size_bytes > MAX_MODEL_BYTES:
    raise RuntimeError(f"Model exceeds 100 MB: {size_bytes:,} bytes. Do not label this export as under 100 MB.")
print("PASS: final GGUF is within the strict 100 MB limit.")

In [ ]:
# Smoke test: generate a short response from the quantized model.
! /content/llama.cpp/build/bin/llama-cli -m /content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf -p "Who are you?" -n 32 --temp 0.2 --no-display-prompt

In [ ]:
# Download the resulting GGUF to your device.
from google.colab import files
files.download("/content/CrazeAi/artifacts/crazeai-1m-q4_k_m.gguf")

## Notes and honest limits

- The model has about **951,000 parameters**. It is randomly initialized and trained from scratch; 10,000 examples do not make it equivalent to a pretrained assistant.
- The ~10-minute target applies to training only and depends on the assigned GPU, sequence lengths, token counts, and compilation. Use the printed wall times as the actual measurement.
- The notebook uses the SmolTalk dataset plus CrazeAI-specific seed/identity examples. Only use data you are permitted to train on, and check the dataset license before redistribution.
- The output-size cell fails if the final GGUF exceeds 100,000,000 bytes.
- If the runtime disconnects or runs out of memory, reduce batch size first; then reduce sequence length. Save artifacts to Google Drive if you need to preserve checkpoints.